<a href="https://colab.research.google.com/github/ilincabaiasu/IB9AU/blob/main/Task_14.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Required Task 14

*Note: GitHub does not support live outputs, please open Colab for running the code.*

### Details

Build a Page-Wise Visual RAG (Retrieval-Augmented Generation) system to analyse AstraZeneca's
FY and Q4 2025 earnings report. Rather than simply reading the PDF as text, your system will
identify the most relevant page for a given query, render it as an image, and use a Vision-Language
Model (VLM) to extract and interpret the information visually — just as a financial analyst would.

This mirrors a real-world analyst workflow: locate the right section of a report, then read it carefully to
extract structured insights.
Setup
Use the notebook structure from the lab session. Your system should use:
Embeddings: sentence-transformers/all-MiniLM-L6-v2 (local, CPU)
VLM: Qwen/Qwen2.5-VL-3B-Instruct (local, T4 GPU)
PDF: AstraZeneca-Q4-2025-earnings.pdf
Runtime: Google Colab with T4 GPU
Tasks

Task 1 — Revenue Table Extraction
Use your Visual RAG system to answer the following query:
"What were AstraZeneca's total Product Sales and Alliance Revenue for FY 2025, and how did each
change compared to FY 2024?"

ask 2 — Regional Revenue Breakdown
Issue the following query:
"Which geographic region had the highest Total Revenue growth in FY 2025, and what was the
growth rate at constant exchange rates?"

Task 3 — R&D Pipeline Interpretation
Issue the following query:
"Which medicines received regulatory approvals in the US between November 2025 and February
2026, and for what indications?"

Hint: Check the python notebook RAG_5_Multimodal_Chunking_OpenSourced.ipynb
for context and further details

What I found interesting:
- No need for external API
- Seems very useful / practical
- 2 AI systems working together, with a seamless outcome

In [ ]:
!pip install -qU Pillow
print

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU detected: {gpu_name} ({vram_gb:.1f} GB VRAM)")
else:
    print("NO GPU DETECTED — go to Runtime → Change runtime type → T4 GPU")
    raise RuntimeError("T4 GPU required.")

!pip install -qU transformers accelerate
!pip install -qU llama-index-core llama-index-readers-file llama-index-embeddings-huggingface pypdf
!pip install -qU pdf2image
!apt-get install -q poppler-utils

print("All dependencies installed.")

### Load Embedding Model



In [ ]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("Embedding model loaded: sentence-transformers/all-MiniLM-L6-v2")

### Load the VLM

In [ ]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

VLM_MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

print(f"⏳ Loading {VLM_MODEL_ID}...")

vlm_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    VLM_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="cuda"   # explicit cuda — avoids silent CPU fallback
)
vlm_processor = AutoProcessor.from_pretrained(VLM_MODEL_ID)

model_device = next(vlm_model.parameters()).device
print(f"VLM loaded on: {model_device}")

### Configure LlamaIndex



In [ ]:
Settings.embed_model = embed_model
Settings.llm = None   # visual QA is handled by Qwen VLM, not LlamaIndex LLM
print("LlamaIndex settings configured.")

Load AstraZeneca PDF

In [ ]:
PDF_FILE = "/content/AstraZeneca-Q4-2025-earnings.pdf"

print(f"   Loading {PDF_FILE}...")
documents = SimpleDirectoryReader(input_files=[PDF_FILE]).load_data()

print(f"   Loaded {len(documents)} pages.")
print(f"   Sample metadata: {documents[0].metadata}")

### Vector Index

In [ ]:
print("Building vector index over all pages...")
index     = VectorStoreIndex.from_documents(documents)
retriever = index.as_retriever(similarity_top_k=1)
print("Index ready — retriever configured for top-1 page.")

### Visual RAG Orchestrator

In [ ]:
from pdf2image import convert_from_path
from PIL import Image
import torch

def query_visual_rag(query_text, max_new_tokens=512, show_page=True):
    """
    Full Visual RAG pipeline:
      1. Semantic search → best page
      2. Render page as image
      3. Qwen2.5-VL reads image and answers query

    Args:
        query_text     : the question to answer
        max_new_tokens : token budget for VLM answer (default 512)
        show_page      : display the rendered page inline (default True)

    Returns:
        answer (str)
    """
    print(f"\n🔍 Query: '{query_text}'")

    # GPU check
    if str(next(vlm_model.parameters()).device) == 'cpu':
        print("⚠️  WARNING: VLM is on CPU — switch to T4 GPU for fast inference.")

    # ── 1. RETRIEVE ──────────────────────────────────────────────────────────
    nodes = retriever.retrieve(query_text)
    if not nodes:
        return "No relevant page found."

    best_node  = nodes[0]
    page_label = best_node.metadata.get('page_label', '1')
    page_index = int(page_label) - 1

    print(f"📍 Best match: Page {page_label}  (similarity score: {best_node.score:.4f})")
    print(f"   Snippet: {best_node.text[:120]}...")

    # ── 2. RENDER ─────────────────────────────────────────────────────────────
    print("Rendering page as image at 150 DPI...")
    pages      = convert_from_path(PDF_FILE,
                                   first_page=page_index + 1,
                                   last_page=page_index + 1,
                                   dpi=150)
    page_image = pages[0]

    if show_page:
        import matplotlib.pyplot as plt
        plt.figure(figsize=(10, 14))
        plt.imshow(page_image)
        plt.title(f"Retrieved: Page {page_label}")
        plt.axis('off')
        plt.tight_layout()
        plt.show()

    # ── 3. VLM ANSWER ─────────────────────────────────────────────────────────
    print(f"🚀 Sending to Qwen2.5-VL (max {max_new_tokens} tokens)...")
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": page_image},
                {"type": "text", "text": (
                    f"You are an expert financial analyst reviewing Page {page_label} "
                    "of AstraZeneca's FY and Q4 2025 Earnings Report. "
                    "Answer the following question based ONLY on the text, tables, and "
                    "charts visible on this page. "
                    "Be precise — include specific numbers, percentages, and table row "
                    "labels exactly as they appear. "
                    "If the answer involves a table, quote the relevant row and column values.\n\n"
                    f"Question: {query_text}"
                )}
            ]
        }
    ]

    text_prompt = vlm_processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = vlm_processor(
        text=[text_prompt],
        images=[page_image],
        return_tensors="pt"
    ).to(vlm_model.device)

    with torch.no_grad():
        output_ids = vlm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
        )

    generated = output_ids[:, inputs['input_ids'].shape[1]:]
    answer    = vlm_processor.batch_decode(generated, skip_special_tokens=True)[0]
    return answer


print("query_visual_rag() ready.")

## TASK 1: Revenue Table Extraction

In [ ]:
from IPython.display import display, Markdown

q1 = (
    "What were AstraZeneca's total Product Sales and Alliance Revenue for FY 2025, "
    "and how did each change compared to FY 2024?"
)

display(Markdown(f"### Task 1 Query\n> {q1}"))
answer_1 = query_visual_rag(q1, max_new_tokens=512)
display(Markdown(f"### Answer\n{answer_1}"))

## TASK 2: Regional Revenue Breakdown

In [ ]:
q2 = (
    "Which geographic region had the highest Total Revenue growth in FY 2025, "
    "and what was the growth rate at constant exchange rates?"
)

display(Markdown(f"### Task 2 Query\n> {q2}"))
answer_2 = query_visual_rag(q2, max_new_tokens=512)
display(Markdown(f"### Answer\n{answer_2}"))

## TASK 3: Pipeline Implementation

In [ ]:
q3 = (
    "Which medicines received regulatory approvals in the US between "
    "November 2025 and February 2026, and for what indications?"
)

display(Markdown(f"### Task 3 Query\n> {q3}"))
answer_3 = query_visual_rag(q3, max_new_tokens=512)
display(Markdown(f"### Answer\n{answer_3}"))

## TASK 4: Audit Mode Query

In [ ]:
q4 = (
    "From AstraZeneca's cash flow statement (Table 12), provide the exact figures for: "
    "(1) Net cash from operating activities for FY 2025 and FY 2024, "
    "(2) Capital expenditure for FY 2025 and FY 2024, and "
    "(3) Free cash flow for FY 2025 and FY 2024. "
    "For each figure, state the table name, the exact row label, and the column heading "
    "exactly as they appear in the document. "
    "Do not estimate — only report numbers visible on this page."
)

display(Markdown(f"### Task 4 Audit Query\n> {q4}"))
answer_4 = query_visual_rag(q4, max_new_tokens=600)
display(Markdown(f"### Answer\n{answer_4}"))